# Наивный Байесовский Классификатор для классификации спам-сообщений

In [1]:
import numpy as np
import pandas as pd

Прочитаем файл (разделителем здесь выступает символ табуляции).

In [4]:
sms_data = pd.read_csv('Data/SMSSpamCollection.csv', header=None, sep='\t', names=['Label', 'SMS'])
sms_data.head()

,Label,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Посмотрим, сколько объектов каждого класса присутствует в датасете.

In [5]:
sms_data.groupby('Label').count()

,SMS
Label,
ham,4825
spam,747


### Предобработка данных

Удаляем символы, не являющиеся буквами, приводим тексты SMS к нижнему регистру, разбиваем строки на слова.

In [8]:
sms_data_clean = sms_data.copy()

In [9]:
# Удаление всех не буквенно-цифровых символов, заменяя их пробелами
sms_data_clean['SMS'] = sms_data_clean['SMS'].str.replace(r'\W+', ' ', regex=True)

# Удаление лишних пробелов (замена нескольких пробелов на один) и удаление пробелов в начале и конце строки
sms_data_clean['SMS'] = sms_data_clean['SMS'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Приведение текста к нижнему регистру
sms_data_clean['SMS'] = sms_data_clean['SMS'].str.lower()

# Разбиение строки на отдельные слова (токенизация)
sms_data_clean['SMS'] = sms_data_clean['SMS'].str.split()

# Вывод первых пяти обработанных SMS-сообщений
sms_data_clean['SMS'].head()

0    [go, until, jurong, point, crazy, available, o...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, in, 2, a, wkly, comp, to, win, f...
3    [u, dun, say, so, early, hor, u, c, already, t...
4    [nah, i, don, t, think, he, goes, to, usf, he,...
Name: SMS, dtype: object

In [11]:
# Считаем процентах количество спам и неспам сообщений
sms_data_clean['Label'].value_counts() / sms_data_clean.shape[0] * 100

Label
ham     86.593683
spam    13.406317
Name: count, dtype: float64

### Разделение на обучающую и тестовую выборки

In [12]:
# Разделение данных на обучающую и тестовую выборки
# Выбираем 80% данных случайным образом для обучающей выборки
train_data = sms_data_clean.sample(frac=0.8, random_state=42)
# Оставшиеся 20% данных используем для тестовой выборки, исключая строки, попавшие в train_data
test_data = sms_data_clean.drop(train_data.index)

# Сброс индексов в обучающей выборке (чтобы они шли подряд)
train_data = train_data.reset_index(drop=True)
# Сброс индексов в тестовой выборке (чтобы они шли подряд)
test_data = test_data.reset_index(drop=True)

In [ ]:
# Считаем в процентах тренировочну выборку
train_data['Label'].value_counts() / train_data.shape[0] * 100

Label
ham     86.698071
spam    13.301929
Name: count, dtype: float64

In [14]:
train_data.shape

(4458, 2)

In [15]:
# Считаем в процентах тестовою (валидационную) выборку
test_data['Label'].value_counts() / test_data.shape[0] * 100

Label
ham     86.175943
spam    13.824057
Name: count, dtype: float64

In [17]:
test_data.shape

(1114, 2)

Мы видим, что и в обучающей, и в тестовой выборке содержится примерно 86-87% спама – как и в нашем оригинальном датасете.

### Список слов

Создаём список всех слов, встречающихся в обучающей выборке.

In [18]:
# Запишем в список все слова из тренировочной выборки без повторений
vocabulary = list(set(train_data['SMS'].sum()))

In [20]:
# Смотрим на срез данных
vocabulary[11:20]

['shitinnit',
 'failing',
 'minecraft',
 'christians',
 'returned',
 'hire',
 'within',
 '24',
 'temp']

In [21]:
# Общее количество слов в списке
len(vocabulary)

7816

### Рассчитаем частоты слов

Для каждого SMS-сообщения посчитаем, сколько раз в нём встречается каждое слово.

In [23]:
# Создание DataFrame, содержащего количество вхождений (частота) каждого слова из словаря в каждом SMS
word_counts_per_sms = pd.DataFrame([
    # Для каждой строки (SMS) в обучающем наборе данных считаем количество вхождений каждого слова из словаря
    [row['SMS'].count(word) for word in vocabulary]  
    for _, row in train_data.iterrows()],  # Перебираем строки обучающей выборки
    columns=vocabulary  # Используем слова из словаря в качестве названий столбцов
)

# Вывод первых строк полученного DataFrame
word_counts_per_sms.head()

,lkpobox177hp51fl,dot,gin,yetty,eat,5free,someone,hunny,browse,versus,...,untamed,nz,qjkgighjjgcbl,cheese,mila,cleaning,heard,01223585236,150ppmsg,permissions
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Добавим частоты каждого слова в обучающий датасет.

In [24]:
train_data = pd.concat([train_data, word_counts_per_sms], axis=1)

In [25]:
train_data.head()

,Label,SMS,lkpobox177hp51fl,dot,gin,yetty,eat,5free,someone,hunny,...,untamed,nz,qjkgighjjgcbl,cheese,mila,cleaning,heard,01223585236,150ppmsg,permissions
0,ham,"[squeeeeeze, this, is, christmas, hug, if, u, ...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,ham,"[and, also, i, ve, sorta, blown, him, off, a, ...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ham,"[mmm, thats, better, now, i, got, a, roast, do...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,ham,"[mm, have, some, kanji, dont, eat, anything, h...",0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ham,"[so, there, s, a, ring, that, comes, with, the...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Значения для формулы Байеса

Посчитаем необходимые значения для формулы Байеса.

In [27]:
# Определим уровень альфа как 1
alpha = 1

In [28]:
# Количество уникальных слов в словаре (размер словарного запаса)
Nvoc = len(vocabulary)

# Вероятность того, что случайное сообщение является спамом
Pspam = train_data['Label'].value_counts()['spam'] / train_data.shape[0]

# Вероятность того, что случайное сообщение не является спамом (то есть "ham")
Pham = train_data['Label'].value_counts()['ham'] / train_data.shape[0]

# Общее количество слов в SMS-сообщениях, которые классифицированы как спам
Nspam = train_data.loc[train_data['Label'] == 'spam', 'SMS'].apply(len).sum()

# Общее количество слов в SMS-сообщениях, которые классифицированы как "ham" (не спам)
Nham = train_data.loc[train_data['Label'] == 'ham', 'SMS'].apply(len).sum()

In [29]:
# Функция для вычисления вероятности появления слова в спам-сообщениях
def p_w_spam(word):
    # Проверяем, есть ли слово в обучающем наборе данных
    if word in train_data.columns:
        # Используем формулу Лапласовского сглаживания для вычисления вероятности
        return (train_data.loc[train_data['Label'] == 'spam', word].sum() + alpha) / (Nspam + alpha * Nvoc)
    else:
        # Если слова нет в словаре, возвращаем 1 (нейтральное значение)
        return 1

# Функция для вычисления вероятности появления слова в не-спам (ham) сообщениях
def p_w_ham(word):
    # Проверяем, есть ли слово в обучающем наборе данных
    if word in train_data.columns:
        # Используем формулу Лапласовского сглаживания для вычисления вероятности
        return (train_data.loc[train_data['Label'] == 'ham', word].sum() + alpha) / (Nham + alpha * Nvoc)
    else:
        # Если слова нет в словаре, возвращаем 1 (нейтральное значение)
        return 1

Лапласовское сглаживание — это метод в вероятностной статистике, который добавляет небольшое значение (обычно 1 или $\alpha$) к числителю и увеличивает знаменатель, чтобы избежать нулевых вероятностей для слов, которые не встречались в обучающих данных.

Используется в наивном байесовском классификаторе, чтобы корректно оценивать вероятность редких или отсутствующих слов в тексте.

Это помогает избежать ситуаций, когда редкие слова обнуляют вероятность всего сообщения.

### Готовим алгоритм классификации

In [30]:
# Функция классификации сообщения как "спам" или "не-спам" (ham)
def classify(message):
    # Начальные вероятности: P(spam) и P(ham) из обучающих данных
    p_spam_given_message = Pspam
    p_ham_given_message = Pham

    # Проходим по каждому слову в сообщении и обновляем вероятности
    for word in message:
        p_spam_given_message *= p_w_spam(word)  # Умножаем на вероятность слова при спаме
        p_ham_given_message *= p_w_ham(word)  # Умножаем на вероятность слова при ham

    # Сравниваем полученные вероятности и классифицируем сообщение
    if p_ham_given_message > p_spam_given_message:
        return 'ham'  # Сообщение скорее не является спамом
    elif p_ham_given_message < p_spam_given_message:
        return 'spam'  # Сообщение скорее является спамом
    else:
        return 'классификация некорректна'  # В редком случае равных вероятностей

### Используем тестовые данные

In [ ]:
# Предсказываем (классифицируем) сообщения на тестовой выборке
test_data['predicted'] = test_data['SMS'].map(classify)

In [33]:
test_data.head()

,Label,SMS,predicted
0,ham,"[u, dun, say, so, early, hor, u, c, already, t...",ham
1,ham,"[nah, i, don, t, think, he, goes, to, usf, he,...",ham
2,spam,"[freemsg, hey, there, darling, it, s, been, 3,...",ham
3,spam,"[had, your, mobile, 11, months, or, more, u, r...",spam
4,ham,"[oh, k, i, m, watching, here]",ham


In [34]:
# Считаем количество данных которые нам удалось классифицировать правильно
correct = (test_data['predicted'] == test_data['Label']).sum() / test_data.shape[0]
print(f"Правильных предсказаний {correct * 100:3f} %")

Правильных предсказаний 98.025135 %


Нам удалось с помощью классификатора реализованного вручную предсказать (классифицировать) тестовые данные с точностью в 98 процентов. Это очень хороший результат.

Далее выводим, какие сообщения не удалось определить корректно (среди оставшихся 2 процента):

In [35]:
test_data.loc[test_data['predicted'] != test_data['Label']].head()

,Label,SMS,predicted
2,spam,"[freemsg, hey, there, darling, it, s, been, 3,...",ham
96,ham,"[waiting, for, your, call]",spam
182,ham,"[26th, of, july]",spam
269,spam,"[sms, ac, jsco, energy, is, high, but, u, may,...",ham
344,ham,"[the, last, thing, i, ever, wanted, to, do, wa...",классификация некорректна


# Наивный байесовский классификатор в sklearn
Ура, мы реализовали наивный байесовский классификатор с нуля!
А теперь посмотрим, как то же самое можно сделать с помощью библиотеки scikit-learn.

In [36]:
# Импортируем необходимые библиотеки из sklearn
# CountVectorizer преобразует текстовые данные в матрицу частот слов (Bag of Words)
from sklearn.feature_extraction.text import CountVectorizer
# accuracy_score вычисляет точность модели, сравнивая предсказанные и реальные метки
from sklearn.metrics import accuracy_score
# train_test_split делит данные на обучающую и тестовую выборки
from sklearn.model_selection import train_test_split
# MultinomialNB — наивный байесовский классификатор для дискретных данных (например, частот слов)
from sklearn.naive_bayes import MultinomialNB

Прочитаем заново csv-файл и предобработаем данные. Разбивать сообщения на слова в этот раз не нужно, мы сделаем это далее с помощью встроенных инструментов

In [37]:
# Загрузка CSV файла в DataFrame без заголовков, разделителя - табуляция, указание имён колонок
df = pd.read_csv(
    "Data/SMSSpamCollection.csv", header=None, sep="\t", names=["Label", "SMS"]
)
# Удаление всех неалфавитных символов (кроме пробела) в столбце 'SMS' и приведение текста к нижнему регистру
df["SMS"] = df["SMS"].str.replace(r"\W+", " ", regex=True).str.lower()
# Замена множества пробелов на один пробел и удаление лишних пробелов в начале и конце строки
df['SMS'] = df['SMS'].str.replace(r'\s+', ' ', regex=True).str.strip()
# Приведение текста в столбце 'SMS' снова к нижнему регистру (для консистентности)
df['SMS'] = df['SMS'].str.lower()
# Вывод первых 5 строк DataFrame для проверки результата
df.head()

,Label,SMS
0,ham,go until jurong point crazy available only in ...
1,ham,ok lar joking wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor u c already then say
4,ham,nah i don t think he goes to usf he lives arou...


Преобразуем строки в векторный вид – то есть, снова создадим таблицу с частотами слов. Но в этот раз воспользуемся встроенным в sklearn классов CountVectorizer().

In [41]:
# Создание экземпляра векторизатора для преобразования текста в числовые векторы (мешок слов)
vectorizer = CountVectorizer()

# Применение векторизатора к столбцу 'SMS' и преобразование текста в числовые представления
X_vectoirized = vectorizer.fit_transform(df["SMS"])

# Присваивание меток (классов) из столбца 'Label' в отдельную переменную y
y = df["Label"]

# Вывод формы матрицы признаков X и меток y для проверки размера данных
print(X_vectoirized.shape, y.shape)

(5572, 8713) (5572,)


С помощью функции `train_test_split` из scikit-learn разобьём выборку на обучающую и тестовую в пропорции 80/20. Не забудем сделать стратификацию!

In [42]:
# Разделение данных на обучающую и тестовую выборки
# X - признаки (векторизованные данные), y - метки (классы)
# test_size=0.2 указывает, что 20% данных пойдут в тестовую выборку
# stratify=y обеспечивает сохранение пропорций классов в обучающей и тестовой выборках
# random_state=1 гарантирует воспроизводимость разбиения данных
X_train, X_test, y_train, y_test = train_test_split(X_vectoirized, y, test_size=0.2, stratify=y, random_state=42)

In [43]:
# Создание экземпляра классификатора Naive Bayes для многоклассовой классификации (MultinomialNB)
clf = MultinomialNB()

# Обучение классификатора на обучающих данных (X_train и y_train)
clf.fit(X_train, y_train)

# Прогнозирование меток на тестовых данных (X_test)
y_test_pred = clf.predict(X_test)

# Вычисление и вывод точности модели, сравнивая предсказания с истинными метками тестовой выборки
print(f"Accuracy: {accuracy_score(y_test, y_test_pred)}")

Accuracy: 0.9829596412556054
